# 03 — Evaluation
**DS593 Final Project**

Evaluates the RAG system against a golden test set derived from ToS;DR annotations.
Compares RAG vs vanilla GPT baseline on:
1. Factual accuracy (yes/no questions with verified answers)
2. Risk rating match (RED/YELLOW/GREEN vs ToS;DR Good/Neutral/Bad labels)


## Step 0: Install Dependencies

In [ ]:
!pip install -q langchain langchain-openai chromadb openai pandas requests scikit-learn
print("✓ Dependencies installed")


## Step 1: Configure API Key

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

if os.getenv("OPENAI_API_KEY") and os.getenv("OPENAI_API_KEY").startswith("sk-"):
    print("✓ OpenAI API key configured")
else:
    raise ValueError("Missing OpenAI API key")


## Step 2: Imports

In [ ]:
import requests
import pandas as pd
import json
from langchain_openai import ChatOpenAI
import chromadb
from chromadb.utils import embedding_functions
from sklearn.metrics import classification_report, confusion_matrix

print("✓ All imports successful")


## Step 3: Load ChromaDB + LLM

In [ ]:
CHROMA_DIR  = "chroma_db"
EMBED_MODEL = "text-embedding-3-small"

openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.environ["OPENAI_API_KEY"],
    model_name=EMBED_MODEL,
)

client     = chromadb.PersistentClient(path=CHROMA_DIR)
collection = client.get_collection(
    name="tos_documents",
    embedding_function=openai_ef,
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.1, max_tokens=500)

print(f"✓ ChromaDB loaded: {collection.count()} chunks")
print(f"✓ LLM initialized")


## Step 4: RAG + Baseline Functions

In [ ]:
def retrieve_chunks(query, company, k=3):
    where = {"company": {"$eq": company}}
    results = collection.query(
        query_texts=[query],
        n_results=k,
        where=where,
    )
    return results["documents"][0]


def rag_query(question, company, k=3):
    """RAG pipeline — retrieve then generate."""
    chunks  = retrieve_chunks(question, company=company, k=k)
    context = "\n\n".join(chunks)

    prompt = f"""You are a legal analyst helping users understand Terms of Service and Privacy Policy documents.
Answer the question based ONLY on the provided document excerpts.
Be direct. End your answer with:
- VERDICT: Yes / No / Unclear
- RISK: RED / YELLOW / GREEN

Document excerpts:
{context}

Question: {question}

Answer:"""

    return llm.invoke(prompt).content


def baseline_query(question, company):
    """Vanilla LLM with no retrieval."""
    prompt = f"""You are a legal analyst. Answer this question about {company}'s Terms of Service or Privacy Policy.
Be direct. End your answer with:
- VERDICT: Yes / No / Unclear
- RISK: RED / YELLOW / GREEN

Question: {question}

Answer:"""
    return llm.invoke(prompt).content


print("✓ Functions ready")


## Step 5: Fetch ToS;DR Ground Truth

ToS;DR provides human-annotated clause ratings for major services.
We fetch their ratings for companies in our corpus and use them as ground truth.
API docs: https://api.tosdr.org


In [ ]:
import json
import time
from collections import defaultdict
from urllib.parse import urlparse

SEARCH_ALIASES = {
    "twitter_x": "twitter",
}


def normalize_host(host):
    host = host.lower().removeprefix("www.")
    if host.startswith("help."):
        host = host[5:]
    return host


def api_get_json(url, retries=4, pause_seconds=1.5):
    """GET JSON from ToS;DR with basic retry handling for rate limits."""
    last_error = None
    for attempt in range(retries):
        try:
            response = requests.get(url, timeout=20)
            if response.status_code == 429:
                time.sleep(pause_seconds * (attempt + 1))
                continue
            response.raise_for_status()
            return response.json()
        except Exception as exc:
            last_error = exc
            time.sleep(pause_seconds * (attempt + 1))
    raise last_error


def build_company_domains(manifest_path="tos_corpus/manifest.json"):
    company_domains = defaultdict(set)
    with open(manifest_path) as f:
        manifest_rows = json.load(f)
    for row in manifest_rows:
        host = normalize_host(urlparse(row["url"]).netloc)
        company_domains[row["company"]].add(host)
    return dict(company_domains)


def domains_match(corpus_host, service_host):
    return (
        corpus_host == service_host
        or corpus_host.endswith(f".{service_host}")
        or service_host.endswith(f".{corpus_host}")
    )


def resolve_tosdr_service(company, company_domains):
    """Find the live ToS;DR service ID that best matches our corpus company."""
    query = SEARCH_ALIASES.get(company, company)
    search_url = f"https://api.tosdr.org/search/v5/?query={query}"
    services = api_get_json(search_url).get("services", [])
    normalized_domains = {normalize_host(domain) for domain in company_domains}

    for service in services:
        service_urls = [normalize_host(url) for url in service.get("urls", [])]
        if any(domains_match(domain, service_url) for domain in normalized_domains for service_url in service_urls):
            return service

    for service in services:
        haystack = " ".join([
            service.get("name", "").lower(),
            service.get("slug", "").lower(),
        ])
        if query.replace("_", " ") in haystack:
            return service

    return None


def fetch_tosdr_points(service_id):
    """Fetch annotated points for a service from the current ToS;DR API."""
    url = f"https://api.tosdr.org/service/v2/?id={service_id}"
    data = api_get_json(url)
    return data.get("parameters", {}).get("points", [])


company_domains = build_company_domains()
tosdr_service_map = {}
all_points = {}

for company, domains in sorted(company_domains.items()):
    print(f"Resolving {company}...")
    try:
        service = resolve_tosdr_service(company, domains)
        if not service:
            print("  No matching ToS;DR service found")
            all_points[company] = []
            continue

        service_id = service["id"]
        tosdr_service_map[company] = {
            "id": service_id,
            "name": service.get("name", company),
            "slug": service.get("slug", ""),
        }
        points = fetch_tosdr_points(service_id)
        all_points[company] = points
        print(f"  Matched ToS;DR service {service_id}: {service.get('name', company)}")
        print(f"  {len(points)} annotated points")
        time.sleep(0.5)
    except Exception as e:
        print(f"  Failed to fetch {company}: {e}")
        all_points[company] = []

print(f"\n✓ Fetched ToS;DR data for {len(all_points)} companies")
print(f"  Matched services: {len(tosdr_service_map)}")


## Step 6: Build Golden Test Set

Convert ToS;DR annotations into Q&A pairs with ground truth answers and risk ratings.
Each point becomes one test case.


In [ ]:
def tosdr_status_to_risk(classification):
    """Map ToS;DR classification to RED/YELLOW/GREEN."""
    mapping = {
        "bad": "RED",
        "neutral": "YELLOW",
        "good": "GREEN",
        "blocker": "RED",
    }
    return mapping.get(str(classification).lower(), "YELLOW")


def build_question_from_point(point):
    """Convert a ToS;DR annotation into a yes/no question."""
    title = (point.get("title") or "").strip().rstrip(".?")
    if not title:
        return None
    return f"Is it true that {title.lower()}?"


golden_test_set = []
seen_point_ids = set()

for company, points in all_points.items():
    for point in points:
        point_id = point.get("id")
        if point_id in seen_point_ids:
            continue

        title = (point.get("title") or "").strip()
        case = point.get("case") or {}
        tosdr_classification = (
            case.get("classification")
            or point.get("classification")
            or point.get("status")
        )

        if not title or not tosdr_classification:
            continue

        # Workflow states like approved/declined are not risk labels.
        if str(tosdr_classification).lower() not in {"bad", "good", "blocker"}:
            continue

        question = build_question_from_point(point)
        if not question:
            continue

        seen_point_ids.add(point_id)
        golden_test_set.append({
            "company": company,
            "question": question,
            "tosdr_title": title,
            "tosdr_status": str(tosdr_classification).lower(),
            "ground_truth_verdict": "Yes",
            "ground_truth_risk": tosdr_status_to_risk(tosdr_classification),
            "tosdr_point_id": point_id,
        })

df_golden = pd.DataFrame(golden_test_set)

if df_golden.empty:
    raise ValueError(
        "Golden test set is empty. Re-run Step 5 and confirm the ToS;DR API returned points with case.classification values."
    )

# Sample up to 5 per company for a balanced test set.
df_golden = pd.concat(
    [
        group.sample(min(len(group), 5), random_state=42)
        for _, group in df_golden.groupby("company")
    ],
    ignore_index=True,
)

print(f"✓ Golden test set: {len(df_golden)} questions")
print(f"  Companies: {df_golden['company'].nunique()}")
print(f"  Risk distribution:\n{df_golden['ground_truth_risk'].value_counts().to_string()}")
print()
print(df_golden[["company", "question", "ground_truth_risk"]].to_string(index=False))


## Step 7: Run Evaluation

Run both RAG and baseline on every question in the golden test set.
Extract VERDICT and RISK from each answer.

**Note:** This makes 2 API calls per question — expect ~2-3 minutes total.


In [ ]:
import re

def extract_verdict(answer):
    """Extract Yes/No/Unclear from model answer."""
    match = re.search(r'VERDICT:\s*(Yes|No|Unclear)', answer, re.IGNORECASE)
    return match.group(1).capitalize() if match else "Unclear"

def extract_risk(answer):
    """Extract RED/YELLOW/GREEN from model answer."""
    match = re.search(r'RISK:\s*(RED|YELLOW|GREEN)', answer, re.IGNORECASE)
    return match.group(1).upper() if match else "YELLOW"


rag_verdicts  = []
rag_risks     = []
base_verdicts = []
base_risks    = []

for i, row in df_golden.iterrows():
    print(f"[{i+1}/{len(df_golden)}] {row['company']} — {row['question'][:60]}...")

    # RAG answer
    rag_answer  = rag_query(row["question"], company=row["company"])
    rag_verdict = extract_verdict(rag_answer)
    rag_risk    = extract_risk(rag_answer)

    # Baseline answer
    base_answer  = baseline_query(row["question"], company=row["company"])
    base_verdict = extract_verdict(base_answer)
    base_risk    = extract_risk(base_answer)

    rag_verdicts.append(rag_verdict)
    rag_risks.append(rag_risk)
    base_verdicts.append(base_verdict)
    base_risks.append(base_risk)

# Add results to dataframe
df_golden["rag_verdict"]  = rag_verdicts
df_golden["rag_risk"]     = rag_risks
df_golden["base_verdict"] = base_verdicts
df_golden["base_risk"]    = base_risks

print("\n✓ Evaluation complete")


## Step 8: Results — Factual Accuracy

In [ ]:
# Factual accuracy — did the model say Yes (confirming the ToS;DR finding)?
# Ground truth is always Yes since ToS;DR confirmed these points exist

rag_correct  = (df_golden["rag_verdict"]  == df_golden["ground_truth_verdict"]).sum()
base_correct = (df_golden["base_verdict"] == df_golden["ground_truth_verdict"]).sum()
total        = len(df_golden)

print("=" * 50)
print("FACTUAL ACCURACY")
print("=" * 50)
print(f"RAG system:  {rag_correct}/{total} = {rag_correct/total*100:.1f}%")
print(f"Baseline:    {base_correct}/{total} = {base_correct/total*100:.1f}%")
print(f"Improvement: +{(rag_correct-base_correct)/total*100:.1f}%")


## Step 9: Results — Risk Rating Accuracy

In [ ]:
rag_risk_correct  = (df_golden["rag_risk"]  == df_golden["ground_truth_risk"]).sum()
base_risk_correct = (df_golden["base_risk"] == df_golden["ground_truth_risk"]).sum()

print("=" * 50)
print("RISK RATING ACCURACY")
print("=" * 50)
print(f"RAG system:  {rag_risk_correct}/{total} = {rag_risk_correct/total*100:.1f}%")
print(f"Baseline:    {base_risk_correct}/{total} = {base_risk_correct/total*100:.1f}%")
print(f"Improvement: +{(rag_risk_correct-base_risk_correct)/total*100:.1f}%")

print("\n── RAG Confusion Matrix (Risk Rating) ──")
print(pd.crosstab(
    df_golden["ground_truth_risk"],
    df_golden["rag_risk"],
    rownames=["Actual"],
    colnames=["Predicted"]
))


## Step 10: Results by Company

In [ ]:
summary = df_golden.groupby("company").apply(lambda x: pd.Series({
    "n":                    len(x),
    "rag_factual_acc":      (x["rag_verdict"]  == x["ground_truth_verdict"]).mean(),
    "base_factual_acc":     (x["base_verdict"] == x["ground_truth_verdict"]).mean(),
    "rag_risk_acc":         (x["rag_risk"]     == x["ground_truth_risk"]).mean(),
    "base_risk_acc":        (x["base_risk"]    == x["ground_truth_risk"]).mean(),
})).reset_index()

summary["factual_delta"] = summary["rag_factual_acc"] - summary["base_factual_acc"]
summary["risk_delta"]    = summary["rag_risk_acc"]    - summary["base_risk_acc"]

print("Results by Company:\n")
print(summary.to_string(index=False, float_format="{:.2f}".format))


## Step 11: Save Results

In [ ]:
df_golden.to_csv("evaluation_results.csv", index=False)
print("✓ Results saved to evaluation_results.csv")
print()
print("=" * 50)
print("FINAL SUMMARY")
print("=" * 50)
print(f"Total test cases:       {total}")
print(f"RAG factual accuracy:   {rag_correct/total*100:.1f}%")
print(f"Base factual accuracy:  {base_correct/total*100:.1f}%")
print(f"RAG risk accuracy:      {rag_risk_correct/total*100:.1f}%")
print(f"Base risk accuracy:     {base_risk_correct/total*100:.1f}%")
